# Heart Disease Dataset — Exploratory Data Analysis

**Task 1 — Individual Component (Wei Cong)**

Dataset: [Heart Failure Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)  
918 patients · 11 clinical features · Binary target (HeartDisease: 0 / 1)

| Step | Analysis |
|------|----------|
| 1 | Dataset overview (shape, dtypes, info) |
| 2 | Missing values & duplicates check |
| 3 | Target variable distribution |
| 4 | Numerical feature distributions & skewness |
| 5 | Categorical feature distributions |
| 6 | Box plots: numerical features by HeartDisease |
| 7 | Heart disease rate by categorical feature |
| 8 | Correlation heatmap (numerical features) |
| 9 | Pairplot of numerical features coloured by target |
| 10 | Cross-tabulation heatmaps (categorical pairs) |
| 11 | Data quality: zero-value detection & imputation |

## Imports

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
print('Imports complete.')

## Step 1 — Load Dataset

In [ ]:
# Path is relative to the notebooks/ directory
df = pd.read_csv('../data/raw/heart.csv')

numerical_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
categorical_features = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
target = 'HeartDisease'

print(f'Shape: {df.shape}')
df.head()

## Step 2 — Data Types & Basic Info

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print()
df.info()

## Step 3 — Missing Values & Duplicates

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print(f'Duplicate rows: {df.duplicated().sum()}')
print()
print('=== Descriptive Statistics ===')
df.describe().round(2)

## Step 4 — Target Variable Distribution

The dataset is slightly imbalanced: ~55% Heart Disease (class 1) vs ~45% No Heart Disease (class 0).
This mild imbalance justifies using **AUC** as the primary evaluation metric rather than Accuracy.

In [ ]:
print('Target variable distribution:')
print(df[target].value_counts())
print(df[target].value_counts(normalize=True).round(3))

plt.figure(figsize=(6, 6))
plt.pie(
    df[target].value_counts(),
    labels=['Heart Disease (1)', 'No Heart Disease (0)'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['#d62728', '#2ca02c']
)
plt.title('Distribution of HeartDisease Target')
plt.tight_layout()
plt.show()

## Step 5 — Numerical Feature Distributions

Histograms with KDE overlays reveal the shape of each numerical variable.
Notably, **Cholesterol** and **RestingBP** show a spike at zero — clinically impossible values
representing missing data, which are corrected in Step 11.

In [ ]:
print(df.describe().round(2))
print()

for feature in numerical_features:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[feature], kde=True, color='steelblue')
    plt.title(f'Distribution of {feature}')
    plt.xlabel(feature)
    plt.tight_layout()
    plt.show()

print('\n=== Skewness ===')
for feature in numerical_features:
    print(f'  {feature}: {df[feature].skew():.3f}')

## Step 6 — Categorical Feature Distributions

In [ ]:
for feature in categorical_features:
    plt.figure(figsize=(8, 4))
    sns.countplot(x=df[feature], palette='muted')
    plt.title(f'Distribution of {feature}')
    plt.xlabel(feature)
    plt.tight_layout()
    plt.show()

## Step 7 — Box Plots: Numerical Features by HeartDisease

Comparing the distribution of each numerical feature between the two classes reveals
which features have the strongest predictive signal. **Oldpeak**, **MaxHR**, and **Age**
show the most distinct separation between classes.

In [ ]:
fig, axes = plt.subplots(1, len(numerical_features), figsize=(20, 5))
for i, col in enumerate(numerical_features):
    df.boxplot(column=col, by=target, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('HeartDisease')
plt.suptitle('Numerical Features by HeartDisease (0=No, 1=Yes)')
plt.tight_layout()
plt.show()

## Step 8 — Heart Disease Rate by Categorical Feature

Bar charts showing the proportion of heart disease cases within each category reveal
the relationship between categorical features and the target:
- **ST_Slope=Flat or Down** → very high disease rate
- **ChestPainType=ASY** → highest disease rate among chest pain types
- **ExerciseAngina=Y** → strongly associated with heart disease

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(categorical_features):
    disease_rate = df.groupby(col)[target].mean()
    disease_rate.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'Heart Disease Rate by {col}')
    axes[i].set_ylabel('Proportion with HeartDisease')
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Step 9 — Correlation Heatmap (Numerical Features)

The correlation matrix shows pairwise linear relationships among numerical features.
**Age** and **MaxHR** show moderate negative correlation (-0.4) — older patients tend to
achieve lower maximum heart rates. Cholesterol shows weak correlations overall.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    df[numerical_features].corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True
)
plt.title('Correlation Matrix — Numerical Features')
plt.tight_layout()
plt.show()

## Step 10 — Pairplot: Numerical Features Coloured by Target

The pairplot reveals multivariate relationships. Each scatter shows a feature pair,
coloured by HeartDisease (blue = no disease, orange = disease).
The diagonal shows KDE distributions per class.

In [ ]:
# Include the target column in the subset so hue='HeartDisease' works correctly
sns.pairplot(
    df[numerical_features + [target]],
    hue=target,
    diag_kind='kde',
    plot_kws={'alpha': 0.5},
    palette={0: 'steelblue', 1: 'salmon'}
)
plt.suptitle('Pair Plot of Numerical Features by HeartDisease', y=1.02)
plt.tight_layout()
plt.show()

## Step 11 — Cross-Tabulation Heatmaps (Categorical Pairs)

Interaction effects between pairs of categorical features, shown as heart disease rate.
Key insight: **Male + ASY chest pain** has the highest disease rate.

In [ ]:
cat_pairs = [
    ('Sex', 'ChestPainType'),
    ('ST_Slope', 'ExerciseAngina'),
    ('Sex', 'ExerciseAngina'),
]

for cat1, cat2 in cat_pairs:
    pivot = df.groupby([cat1, cat2])[target].mean().unstack()
    plt.figure(figsize=(8, 4))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Reds', vmin=0, vmax=1)
    plt.title(f'Heart Disease Rate: {cat1} vs {cat2}')
    plt.tight_layout()
    plt.show()

## Step 12 — Data Quality: Zero Value Detection & Imputation

**Finding:** 172 patients have Cholesterol = 0 and 1 patient has RestingBP = 0.
These are clinically impossible values — a patient cannot have zero serum cholesterol
or zero resting blood pressure while alive.

**Action:** Replace zero values with the **median of non-zero values** to preserve the
feature distribution without introducing artificial outliers. This imputation is also
applied in the training pipeline (`mlops_assignment/dataset.py`).

In [ ]:
# Zero-value detection
print('=== Zero Values in Numerical Features ===')
for col in numerical_features:
    n_zeros = (df[col] == 0).sum()
    if n_zeros > 0:
        print(f'  {col}: {n_zeros} zero values detected')

print()
print(f'Cholesterol zero count : {(df["Cholesterol"] == 0).sum()}')
print(f'RestingBP  zero count  : {(df["RestingBP"] == 0).sum()}')

# Impute: replace with non-zero median
df_clean = df.copy()
for col in ('Cholesterol', 'RestingBP'):
    n_zeros = (df_clean[col] == 0).sum()
    if n_zeros > 0:
        median_val = df_clean.loc[df_clean[col] > 0, col].median()
        df_clean[col] = df_clean[col].replace(0, median_val)
        print(f'  {col}: {n_zeros} zeros replaced with median = {median_val}')

print()
print('Post-imputation statistics:')
df_clean[['Cholesterol', 'RestingBP']].describe().round(2)

## Summary of Key EDA Findings

| Finding | Implication for Modelling |
|---------|---------------------------|
| 55/45 class split | Mild imbalance — use **AUC** as primary metric, not Accuracy |
| 172 zero Cholesterol values | Impute with non-zero median before training |
| ST_Slope and ExerciseAngina most predictive | Confirm in feature importance post-training |
| Age and MaxHR show clear class separation (box plots) | Good candidate features |
| Weak pairwise correlations among numerical features | Low multicollinearity risk |
| Oldpeak has right skew | Binning or normalisation will reduce outlier impact |

These insights directly informed the preprocessing choices in `configs/config_heart.yaml`:
- **AUC** as evaluation metric
- **Z-score normalisation** (handles skew)
- **Binning** of Age, Cholesterol, RestingBP, MaxHR (aligns with clinical thresholds)
- **Multicollinearity removal** at 0.85 threshold (conservative, preserves predictive features)